# Tugas Proyek Akhir Pengenalan Pola

### "KLASTIK: Sistem Klasifikasi Sampah Plastik
### Kantin FMIPA UGM Berbasis Machine Learning"
#### Disusun Oleh:
- Ahmad Fattah Wardoyo (22/493136/PA/21166)
- Alan Ghani Haris (22/497605/PA/21433)
- Ignatius Gerald Handono (22/494097/PA/21248)
- Ilyaz Raukhillah Adzaqi (22/503561/PA/21646)
- Rafael Andika Radya Hadi Kusuma (22/493103/PA/21164)
- Walid Syaadan Sulthoni (22/506047/PA/21781)

(Keterangan: mohon maaf kami tidak menyediakan file README,
langkah menjalankan program ada di file IPYNB ini, terima kasih)

## 1. Import Library

Anda dapat melakukan instalasi berdasarkan
file requirements.txt terlampir

In [ ]:
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import hog, local_binary_pattern
from imblearn.over_sampling import SMOTE

## 2. Dataset

Lakukan pengunduhan dataset melalui link Google Drive berikut:
http://bit.ly/4kBCNkN

In [ ]:
# ubah sesuai dengan path dataset Anda
folder_path = (
    '/home/rafael/Documents/Kuliah/pattern_recognition/'
    'proyek_akhir/dataset-sampah/dataset-plastic'
)

# options: 'hog', 'lbp', 'hog+color', 'hog+lbp', 'all_combined'
use_feature = 'all_combined'

## 3. Preprocessing Data

Beberapa definisi fungsi untuk tahapan preprocessing

In [ ]:
# Label
def get_label_from_filename(filename):
    name = os.path.splitext(filename)[0]
    label = re.split(r'\d+', name)[0]
    return label.lower()

# Preprocessing
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (128, 128))
    eq = cv2.equalizeHist(resized)
    blur = cv2.GaussianBlur(eq, (5, 5), 0)
    return blur

## 4. Feature Extraction and Selection

### Feature Extraction

In [ ]:
def extract_hog_features(image):
    features, _ = hog(image, orientations=9,
                      pixels_per_cell=(8, 8),
                      cells_per_block=(2, 2),
                      visualize=True,
                      channel_axis=None)
    return features

def extract_color_hist(image, bins=(8, 8, 8)):
    resized = cv2.resize(image, (128, 128))
    hist = cv2.calcHist([resized], [0, 1, 2], None, bins,
                        [0, 256, 0, 256, 0, 256])
    return cv2.normalize(hist, hist).flatten()

def extract_lbp_features(gray_image, P=8, R=1, method='uniform'):
    lbp = local_binary_pattern(gray_image, P, R, method)
    (hist, _) = np.histogram(lbp.ravel(),
                             bins=np.arange(0, P + 3),
                             range=(0, P + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    return hist

### Feature Selection

In [ ]:
def extract_features(image_path, mode='hog'):
    gray_image = preprocess_image(image_path)
    color_image = None
    if 'color' in mode or mode == 'all_combined':
        color_image = cv2.imread(image_path)
        if color_image is None:
            raise ValueError(f"Error read: {image_path}")

    if mode == 'hog':
        return extract_hog_features(gray_image)
    elif mode == 'color':
        if color_image is None:
            color_image = cv2.imread(image_path)
        return extract_color_hist(color_image)
    elif mode == 'lbp':
        return extract_lbp_features(gray_image)
    elif mode == 'hog+color':
        if color_image is None:
            color_image = cv2.imread(image_path)
        return np.hstack([
            extract_hog_features(gray_image),
            extract_color_hist(color_image)
        ])
    elif mode == 'hog+lbp':
        return np.hstack([
            extract_hog_features(gray_image),
            extract_lbp_features(gray_image)
        ])
    elif mode == 'all_combined':
        if color_image is None:
            color_image = cv2.imread(image_path)
        return np.hstack([
            extract_hog_features(gray_image).flatten(),
            extract_color_hist(color_image).flatten(),
            extract_lbp_features(gray_image).flatten(),
        ])
    else:
        raise ValueError(f"Unsupported feature mode: {mode}")

## 5. Load Data

In [ ]:
features, labels = [], []
image_files = [
    f for f in os.listdir(folder_path)
    if f.endswith(('.png', '.jpg', '.jpeg'))
]

for fname in image_files:
    label = get_label_from_filename(fname)
    full_path = os.path.join(folder_path, fname)
    feat = extract_features(full_path, mode=use_feature)
    features.append(feat)
    labels.append(label)

# print(features)
# print(labels)

### Plotting

In [ ]:
# Labels Distribution
label_counts = {}
for label in labels:
    label_counts[label] = label_counts.get(label, 0) + 1

# Plotting Data
labels_plot = list(label_counts.keys())
counts_plot = list(label_counts.values())

plt.figure(figsize=(6, 7))
bars = plt.bar(labels_plot, counts_plot, color='skyblue')
plt.xlabel('Label')
plt.ylabel('Total Count')
plt.title('Distribusi Label')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             yval,
             int(yval),
             ha='center',
             va='bottom')

plt.show()

# Addition
print("Exact total value for each label:")
for label, count in label_counts.items():
    print(f"=> {label}: {count}")

## 6. Train/Test Split

In [ ]:
# Encode Labels
le = LabelEncoder()
y = le.fit_transform(labels)
X_original = np.array(features)

# Ensure X_original is 2D
if X_original.ndim == 1:
    X_original = np.vstack(X_original).T

# Train/Test Split
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_original, y, stratify=y, test_size=0.2, random_state=42
)

# Scaling Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_orig)
X_test = scaler.transform(X_test_orig)

unique_labels_in_test = np.unique(y_test)
target_names_in_test = le.inverse_transform(unique_labels_in_test)

# Handle Imbalanced Data (for training)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

### Plotting

In [ ]:
def plot_distribution(y_data, title, le):
    unique_labels, counts = np.unique(y_data, return_counts=True)
    labels = le.inverse_transform(unique_labels)
    plt.figure(figsize=(8, 6))
    plt.bar(labels, counts)
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()

plot_distribution(y_train_bal, "Training Data Distribution", le)
plot_distribution(y_test, "Test Data Distribution", le)

## 7. Model ML

In [ ]:
models_to_tune = {
    "SVM (Linear)": (SVC(probability=True), {
        'kernel': ['linear'],
        'C': [0.1, 1, 10, 100]
    }),
    "SVM (RBF)": (SVC(probability=True), {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10, 100],
        'gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto']
    }),
    "Random Forest": (RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }),
    "KNN": (KNeighborsClassifier(), {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    })
}

## 8. Training dan Evaluasi Model

In [ ]:
best_models = {}

for name, (model, params) in models_to_tune.items():
    print(f"\n\nTuning and Evaluating model: {name}")
    grid_search = GridSearchCV(
        model,
        params,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train_bal, y_train_bal)

    print(f"Best parameters for {name}: "
        f"{grid_search.best_params_}")
    print(f"Best cross-validation accuracy for {name}: "
        f"{grid_search.best_score_:.4f}")

    best_model = grid_search.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\nTest Accuracy for {name} (with best params): "
        f"{acc:.4f}")

    current_unique_labels_in_test = np.unique(y_test)
    current_target_names_in_test = le.inverse_transform(
        current_unique_labels_in_test
    )

    print(classification_report(
        y_test,
        y_pred,
        labels=current_unique_labels_in_test,
        target_names=current_target_names_in_test,
        zero_division=0)
    )

    cm = confusion_matrix(y_test,
                          y_pred,
                          labels=current_unique_labels_in_test)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=current_target_names_in_test
    )
    disp.plot(xticks_rotation='vertical', cmap='Blues')
    plt.title(f"Confusion Matrix: {name} (Best Params)")
    plt.tight_layout()
    plt.show()

## 9. Save Model as PKL

In [ ]:
save_model_path = "best_model.pkl"
save_scaler_path = "scaler.pkl"
save_encoder_path = "label_encoder.pkl"

with open(save_model_path, 'wb') as f:
    pickle.dump(best_model, f)
with open(save_scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
with open(save_encoder_path, 'wb') as f:
    pickle.dump(le, f)

## 10. Prediction Mode

Untuk menggunakan aplikasi, lakukan langkah berikut:
- Clone GitHub repository melalui link berikut:
https://github.com/walidtoni/pattern-recog-final-project
- Lakukan instalasi library melalui file requirements.txt tersedia
- Jalankan dengan => $ python app.py
- Buka localhot:5000 di browser Anda

Jika Anda hanya ingin melakukan
cek berbasis kode Python sederhana,
gunakan kode di bawah ini:

In [ ]:
def load_model_components():
    with open(save_model_path, 'rb') as f:
        model = pickle.load(f)
    with open(save_scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    with open(save_encoder_path, 'rb') as f:
        le = pickle.load(f)
    return model, scaler, le

def predict_new_image(image_path,
                      model,
                      scaler,
                      le,
                      mode='all_combined'):
    feat = extract_features(image_path, mode=mode).reshape(1, -1)
    feat_scaled = scaler.transform(feat)
    pred_encoded = model.predict(feat_scaled)[0]
    label = le.inverse_transform([pred_encoded])[0]
    return label

# Example usage
model, scaler, le = load_model_components()
image_path = "" # ubah sesuai dengan path Anda
predicted_label = predict_new_image(image_path,
                                    model,
                                    scaler,
                                    le,
                                    mode='all_combined')
print(f"Predicted: {predicted_label}")